In [8]:
from models.rdunet import RDUNet 

In [4]:
import torch
import importlib
import torch.nn as nn
import torch.optim as optim
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, random_split
import os
from pytorch_msssim import ssim
from torchmetrics.image import PeakSignalNoiseRatio
import matplotlib.pyplot as plt
import numpy as np

In [5]:
device = torch.device('mps' if  torch.backends.mps.is_available() else 'cpu')
print(device)

mps


In [6]:
from base import load_pkl, transform_1, transform_3, calc_loss, CustomData, load_data, fineTune, test_func, NoiseImage
from base import augment, test_func_batches, list_images, collect_images, augment_data


psnr_metric = PeakSignalNoiseRatio(data_range=1.0)
psnr_metric = psnr_metric.to(device)



In [7]:
import base
from base import load_pkl, transform_1, transform_3, calc_loss, CustomData, load_data, fineTune
importlib.reload(base)
psnr_metric = PeakSignalNoiseRatio(data_range=1.0)

In [9]:
rdunet = RDUNet(**{'channels': 1, 'base filters': 128})
rdunet_model = rdunet.to(device)

# Load the .pth checkpoint
checkpoint = torch.load('/Users/tjsss/Desktop/bharatAtomic/semPhase1/model_weights/model_gray.pth', map_location=device)

# Handle two common .pth formats:
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    rdunet_model.load_state_dict(checkpoint['model_state_dict'])
else:
    rdunet_model.load_state_dict(checkpoint)

print("Pretrained weights loaded successfully!")

Pretrained weights loaded successfully!


In [11]:
for name, param in rdunet_model.named_parameters():
    print(f"{name}")

input_block.conv_1.weight
input_block.conv_1.bias
input_block.conv_2.weight
input_block.conv_2.bias
input_block.actv_1.weight
input_block.actv_2.weight
block_0_0.conv_0.weight
block_0_0.conv_0.bias
block_0_0.conv_1.weight
block_0_0.conv_1.bias
block_0_0.conv_2.weight
block_0_0.conv_2.bias
block_0_0.conv_3.weight
block_0_0.conv_3.bias
block_0_0.actv_0.weight
block_0_0.actv_1.weight
block_0_0.actv_2.weight
block_0_0.actv_3.weight
block_0_1.conv_0.weight
block_0_1.conv_0.bias
block_0_1.conv_1.weight
block_0_1.conv_1.bias
block_0_1.conv_2.weight
block_0_1.conv_2.bias
block_0_1.conv_3.weight
block_0_1.conv_3.bias
block_0_1.actv_0.weight
block_0_1.actv_1.weight
block_0_1.actv_2.weight
block_0_1.actv_3.weight
down_0.conv.weight
down_0.conv.bias
down_0.actv.weight
block_1_0.conv_0.weight
block_1_0.conv_0.bias
block_1_0.conv_1.weight
block_1_0.conv_1.bias
block_1_0.conv_2.weight
block_1_0.conv_2.bias
block_1_0.conv_3.weight
block_1_0.conv_3.bias
block_1_0.actv_0.weight
block_1_0.actv_1.weight
b

In [ ]:
to_freeze = ['block_3_1.',
'block_3_0.',
'block_2_1.',
'block_2_0.',
'down_2.',
'block_1_1.',
'block_1_0.',
'down_1.',
'block_0_1.',
'block_0_0.',
'down_0',
'input_block.',]


In [ ]:
for name, param in rdunet_model.named_parameters():
    if any(name.startswith(l) for l in to_freeze):
        param.requires_grad = False
    else:
        param.requires_grad = True

print("After freezing encoder layers:")
print("Trainable parameters:")
for name, param in rdunet_model.named_parameters():
    if param.requires_grad:
        print(f"{name}: requires_grad={param.requires_grad}")
print("\nFrozen parameters:")
for name, param in rdunet_model.named_parameters():
    if not param.requires_grad:
        print(f"{name}: requires_grad={param.requires_grad}")


NameError: name 'to_freeze' is not defined